# izulu smoke test

Exercises the primary API of `izulu`: declarative `Error` subclasses with templated messages, typed fields, and feature toggles.

In [ ]:
# Example 1: Minimal Error subclass with a static template.
from izulu import root

class SimpleError(root.Error):
    __template__ = "something went wrong"

err = SimpleError()
print(repr(err))
print(str(err))
assert isinstance(err, Exception)
assert str(err) == "something went wrong"

In [ ]:
# Example 2: Templated Error with formatted fields (current izulu forbids
# unannotated template fields by default, so annotate them).
from izulu import root

class NotFoundError(root.Error):
    __template__ = "resource {name!r} (id={resource_id}) not found"
    name: str
    resource_id: int

err = NotFoundError(name="widget", resource_id=42)
print(str(err))
assert "'widget'" in str(err)
assert "42" in str(err)
assert err.as_kwargs()["name"] == "widget"
assert err.as_kwargs()["resource_id"] == 42

In [ ]:
# Example 3: Typed fields via class annotations.
from izulu import root

class HTTPError(root.Error):
    __template__ = "HTTP {status}: {reason}"
    status: int
    reason: str

err = HTTPError(status=503, reason="Service Unavailable")
print(str(err))
assert err.as_kwargs() == {"status": 503, "reason": "Service Unavailable"}
assert "503" in str(err)

In [ ]:
# Example 4: Default values via class-level attributes.
from izulu import root

class TimeoutError_(root.Error):
    __template__ = "timed out after {seconds}s ({stage})"
    seconds: float = 30.0
    stage: str = "connect"

default_err = TimeoutError_()
custom_err = TimeoutError_(seconds=5.0, stage="read")
print(str(default_err))
print(str(custom_err))
assert "30.0" in str(default_err)
assert "5.0" in str(custom_err)
assert "read" in str(custom_err)

In [ ]:
# Example 5: Catching izulu errors as ordinary exceptions + as_dict serialization.
from izulu import root

class ValidationError(root.Error):
    __template__ = "field {field!r} invalid: {reason}"
    field: str
    reason: str

caught = None
try:
    raise ValidationError(field="email", reason="missing '@'")
except ValidationError as e:  # an izulu error is caught like any other exception class
    caught = e
else:
    raise AssertionError("expected ValidationError")

assert isinstance(caught, ValidationError)
assert isinstance(caught, Exception)
payload = caught.as_dict()
print(payload)
assert payload["field"] == "email"
assert payload["reason"] == "missing '@'"
print("izulu smoke test passed")